# Predicting Short-Term Stock Market Movements Using Financial News Sentiment
**CIS 635 · Data Mining and Knowledge Discovery · Fall 2026**

**Research question:** Does news sentiment improve next-day directional prediction beyond historical price and volume features?

Pipeline: GDELT headlines + Stooq daily OHLCV → cleaning & session alignment → FinBERT sentiment → feature engineering → LR / DT / RF / SVM → market-only vs market+sentiment on a chronological held-out test set.

Set `SOURCE = "synthetic"` to check the code end to end without downloading anything (synthetic results say nothing about real markets).

In [ ]:
# Colab setup: clone the project and install it (transformers/torch are preinstalled in Colab).
import os, sys
if "google.colab" in sys.modules and not os.path.exists("ST"):
    !git clone https://github.com/collokd/st.git ST
if os.path.exists("ST"):
    %cd ST
%pip install -q -e .

In [ ]:
from pathlib import Path
import pandas as pd
from newsstock.config import Config
from newsstock import pipeline, reporting
from newsstock.features import MARKET_FEATURES, SENTIMENT_FEATURES, build_dataset
from newsstock.modeling import chronological_split, run_experiment
from newsstock.sentiment import aggregate_sentiment, get_scorer, score_headlines
from newsstock.sessions import map_to_sessions

SOURCE = "synthetic"      # "online" (GDELT + Stooq), "csv" (data/raw/{market,news}_<T>.csv) or "synthetic"
USE_FINBERT = True
cfg = Config(tickers=["AAPL", "MSFT", "AMZN", "TSLA", "NVDA"], start="2022-01-01", end="2024-12-31")
cfg.ensure_dirs()

## 1. Acquire and clean data
Duplicate/missing records are removed; headlines must match the company's name pattern to reduce ambiguous ticker matches.

In [ ]:
market, news = pipeline.load_data(cfg, SOURCE)
print(market.groupby("ticker")["date"].agg(["min", "max", "count"]))
news.groupby("ticker").size().rename("headlines")

## 2. Align headlines with trading sessions
A headline published before 16:00 ET on a trading day belongs to that session; later, on weekends or holidays it rolls to the next session. Features for session *t* are known at *t*'s close; the label is the direction of close *t+1* vs close *t*.

In [ ]:
news = map_to_sessions(news, market)
news[["ticker", "published_utc", "session", "headline"]].head()

## 3. FinBERT sentiment scoring and session aggregation

In [ ]:
scorer = get_scorer(USE_FINBERT, cfg.finbert_model)
scored = score_headlines(news, scorer)
print("scorer:", scorer.name)
print(scored["sent_label"].value_counts(normalize=True))
sentiment = aggregate_sentiment(scored)
sentiment.head()

## 4. Feature engineering and EDA

In [ ]:
dataset = build_dataset(market, sentiment)
print(f"{len(dataset):,} ticker-sessions, up-day share {dataset['target'].mean():.3f}, sessions with news {dataset['has_news'].mean():.1%}")
reporting.plot_eda(dataset, cfg.output_dir / "eda.png")
from IPython.display import Image, display
display(Image(cfg.output_dir / "eda.png"))

## 5. Chronological split, tuning on validation, held-out test evaluation

In [ ]:
split = chronological_split(dataset, cfg.train_frac, cfg.val_frac)
for name, part in (("train", split.train), ("val", split.val), ("test", split.test)):
    print(f"{name:5s} {part['date'].min():%Y-%m-%d} .. {part['date'].max():%Y-%m-%d}  n={len(part):,}")

result = run_experiment(split, {"market_only": MARKET_FEATURES,
                                "market_plus_sentiment": MARKET_FEATURES + SENTIMENT_FEATURES}, cfg.random_state)
result.metrics.round(3)

## 6. Market-only vs market + sentiment

In [ ]:
display(reporting.comparison_table(result).round(3))
reporting.plot_metric_comparison(result, cfg.output_dir / "accuracy_comparison.png", "accuracy")
reporting.plot_confusions(result, cfg.output_dir / "confusion_matrices.png")
display(Image(cfg.output_dir / "accuracy_comparison.png"))
display(Image(cfg.output_dir / "confusion_matrices.png"))

## 7. (Optional) Exploratory k-means clustering

In [ ]:
from newsstock.clustering import kmeans_profile
scores, profile = kmeans_profile(split.train, ["ret_1", "vol_5", "volume_chg", "sent_net_mean", "news_count_log"], random_state=cfg.random_state)
display(scores.round(3)); profile.round(3)